# Multiple Sticker Model
Based off Indei & Takimoto (2010), we're gonna implement what they did and compare it to our model. The association and dissociation rates we will have to calculate ourselves from the simulations, so we can start with getting the same plots as they generate.

We make the same assumption that the association and dissociation rates of all the stickers is the same, $\alpha, \beta$

## $\mathbf{K}(l)$ recursively generating $\mathbf W(N_a)$

The relation to generate the transition rate matrix $\mathbf W$ recursively is 
$$
\mathbf K (1) \equiv \begin{pmatrix}
1 & \beta_{0} \\
\alpha_{0} & 1
\end{pmatrix}
$$
$$
\mathbf{K}(l)=\begin{pmatrix}
\mathbf{K}(l-1) & \beta_{(l-1)M}\mathbf{I}_{2^{l-1}} \\
\alpha_{(l-1)M}\mathbf{I}_{2^{l-1}} & \mathbf{K}(l-1)
\end{pmatrix}
$$
$$
W_{s^\prime,s}(N_{a}) = \begin{cases}
K_{s^\prime,s}(N_{a}) & \text{(if $s^\prime \neq s$)}\\
-\sum_{s^{\prime\prime}(\neq s)=0}^{s_{\text{max}}}K_{s^{\prime\prime},s}(N_{a}) & \text{(if $s^\prime = s$)}
\end{cases}
$$

## Generating $\mathbf{K}(l)$

In [6]:
import numpy as np

def recursive_K(l, α, β):
    if l == 1:
        return np.array([[1, β],[α, 1]])
    else:
        tl = recursive_K(l-1, α, β)
        tr = β * np.identity(2 ** (l-1))
        bl = α * np.identity(2 ** (l-1))
        br = recursive_K(l-1, α, β)
        return np.block([[tl, tr],
                          [bl, br]])

def recursive_K_list(l, α_list, β_list):
    if l == 1:
        return np.array([[1, β_list[0]],[α_list[0], 1]])
    else:
        tl = recursive_K_list(l-1, α_list[:-1], β_list[:-1])
        tr = β_list[l-1] * np.identity(2 ** (l-1))
        bl = α_list[l-1] * np.identity(2 ** (l-1))
        br = recursive_K_list(l-1, α_list[:-1], β_list[:-1])
        return np.block([[tl, tr],
                          [bl, br]])


In [7]:
l = 3
α = 2
β = 3
K_l = recursive_K(l, α, β)
# np.identity(int(2 ** (l-1)))
print(K_l)
np.sum(K_l, axis=0)

[[1. 3. 3. 0. 3. 0. 0. 0.]
 [2. 1. 0. 3. 0. 3. 0. 0.]
 [2. 0. 1. 3. 0. 0. 3. 0.]
 [0. 2. 2. 1. 0. 0. 0. 3.]
 [2. 0. 0. 0. 1. 3. 3. 0.]
 [0. 2. 0. 0. 2. 1. 0. 3.]
 [0. 0. 2. 0. 2. 0. 1. 3.]
 [0. 0. 0. 2. 0. 2. 2. 1.]]


array([ 7.,  8.,  8.,  9.,  8.,  9.,  9., 10.])

In [8]:
l = 3
α = np.arange(1, l+1) * 2
β = np.arange(1, l+1) * 3
K_l = recursive_K_list(l, α, β)
print(K_l)

[[1. 3. 6. 0. 9. 0. 0. 0.]
 [2. 1. 0. 6. 0. 9. 0. 0.]
 [4. 0. 1. 3. 0. 0. 9. 0.]
 [0. 4. 2. 1. 0. 0. 0. 9.]
 [6. 0. 0. 0. 1. 3. 6. 0.]
 [0. 6. 0. 0. 2. 1. 0. 6.]
 [0. 0. 6. 0. 4. 0. 1. 3.]
 [0. 0. 0. 6. 0. 4. 2. 1.]]


## Generating $\mathbf W(N_a)$

In [10]:
# N_a is the total number of stickers on a chain
def generate_W(N_a, α, β):
    K_N_a = recursive_K(N_a, α, β)
    K_upper = np.triu(K_N_a, 1)
    K_lower = np.tril(K_N_a, -1)
    
    W_diagonal = np.diagflat(-np.sum(K_N_a, axis=0) + 1)

    return K_upper + K_lower + W_diagonal

def generate_W_lists(N_a, α_list, β_list):
    if len(α_list) != len(β_list) or len(α_list) != N_a:
        raise ValueError("Incorrect number of association/dissociation parameters.")
    K_N_a = recursive_K_list(N_a, α_list, β_list)
    K_upper = np.triu(K_N_a, 1)
    K_lower = np.tril(K_N_a, -1)
    
    W_diagonal = np.diagflat(-np.sum(K_N_a, axis=0) + 1)

    return K_upper + K_lower + W_diagonal


In [11]:
l = 3
α = 2
β = 3

# from equation 14
test_W_3 = np.array([[-6, β, β, 0, β, 0, 0, 0],
                    [α, -7, 0, β, 0, β, 0, 0],
                    [α, 0, -7, β, 0, 0, β, 0],
                    [0, α, α, -8, 0, 0, 0, β],
                    [α, 0, 0, 0, -7, β, β, 0],
                    [0, α, 0, 0, α, -8, 0, β],
                    [0, 0, α, 0, α, 0, -8, β],
                    [0, 0, 0, α, 0, α, α, -9]])

W_3 = generate_W(3, 2, 3)
W_3_list = generate_W_lists(3, [2, 2, 2], [3, 3, 3])

# test if all entries are the same
print(np.array_equal(test_W_3, W_3))
print(np.array_equal(test_W_3, W_3_list))

True
True


# Connectivity matrix $\mathbf B^s$

This matrix determines the equations of motion for an association state $s$, so we should start by decoding $s$ to be an array of association states ($n^s_i = 1$ means $i\text{th}$ bead in association state $s$ is associated, $n^s_i = 0$ is free).

The definition in the paper is a little confusing -- when first defined, $n^s_i$ is implied to be from $i=1,\dots, N_b -1$. However, in the same paragraph, it's stated that for $s=s_\text{max}, n_i^{s_\text{max}}=1$ for all $i$, which is only true for $M=1$ or if $i=1, \dots, N_a$. The latter is likely the error, as there are more usages of the first definition.

This affects extracting $n_i^s$ as we'll pad the gaps between stickers with $M-1$ entries of $0$. 

In [54]:
def extract_n_s(s, M, N_a):
    if N_a < M:
        raise ValueError("N_a must be greater than or equal to M.")
    
    n_s = np.zeros(M * (N_a-1) + 1, dtype=int)

    index = 0
    while True:
        n_s[index] = s & 1
        s = s >> 1
        if s == 0:
            break
        if index >= M * (N_a-1):
            raise ValueError(f"s is too large to fit in the specified number of stickers and spacing, s^max = {2 ** N_a - 1}.")
        index += M
    return n_s

s = 15
N_a = 4
M = 3
print(extract_n_s(s, M, N_a))


[1 0 0 1 0 0 1 0 0 1]


The resulting $\mathbf B^s$ matrix should look like
$$
B_{i,j}^s \equiv a_i^s \delta_{i, j+1} + b_i^s \delta_{i, j} + c_i^s \delta_{i, j-1}
$$
so we have a tridiagonal matrix. These tridiagonal elements are 
$$
(a_i^s, b_i^s, c_i^s) = 
\begin{cases}
(0,0,0) & \text{(if $(n_{i-1}^s,n_i^s)=(1,1)$)} \\
(-1,1,0) & \text{(if $(n_{i-1}^s,n_i^s)=(0,1)$)} \\
(0,1,-1) & \text{(if $(n_{i-1}^s,n_i^s)=(1,0)$)} \\
(-1,2,-1) & \text{(if $(n_{i-1}^s,n_i^s)=(0,0)$)}
\end{cases}
$$
Note that the first case is not possible unless $M=1$ and we have only sticker beads. 

We can set each diagonal individually and compose them together.

In [55]:
def generate_B(N_b, s, M):
    n_s = extract_n_s(s, M)
    B = np.zeros((N_b-1, N_b-1))

    for i, pair in enumerate(zip(n_s[:-1], n_s[1:])):
        if pair == (0, 0):
            # (0, 0, 0)
            if i == 0:
                B[i, i] = 0
                B[i, i+1] = 0
            elif i == N_b - 2:
                B[i, i-1] = 0
                B[i, i] = 0
            else:
                B[i, i-1] = 0
                B[i, i] = 0
                B[i, i+1] = 0
        elif pair == (0, 1):
            # (-1, 1, 0)
            if i == 0:
                B[i, i] = -1
                B[i, i+1] = 0
            elif i == N_b - 2:
                B[i, i-1] = -1
                B[i, i] = 1
            else:
                B[i, i-1] = -1
                B[i, i] = 1
                B[i, i+1] = 0
        elif pair == (1, 0):
            # (0, 1, -1)
            if i == 0:
                B[i, i] = 1
                B[i, i+1] = -1
            elif i == N_b - 2:
                B[i, i-1] = 0
                B[i, i] = 1
            else:
                B[i, i-1] = 0
                B[i, i] = 1
                B[i, i+1] = -1
        elif pair == (1, 1):
            # (-1, 2, -1)
            if i == 0:
                B[i, i] = 2
                B[i, i+1] = -1
            elif i == N_b - 2:
                B[i, i-1] = -1
                B[i, i] = 2
            else:
                B[i, i-1] = -1
                B[i, i] = 2
                B[i, i+1] = -1
    return B

Absolutely awful stuff, let's do this better.

In [107]:
# Note: negating 0 gives -1
def generate_B(s, M, N_b):
    if (N_b-1) % M != 0:
        print("N_b-1:", N_b-1)
        print("M:", M)
        print("N_b-1 % M:", (N_b-1) % M)
        raise ValueError("N_b-1 must be divisible by M.")
    N_a = (N_b - 1) // M + 1
    n_s = extract_n_s(s, M, N_a)
    print("Extracted n_s:", n_s)
    B = np.zeros((N_b-1, N_b-1))

    for i, pair in enumerate(zip(n_s[:-1], n_s[1:])):
        print("Setting row", i, "for pair", pair)
        if i != 0: # a_i
            print("  Setting a_i for i =", i)
            B[i, i-1] = -1 * int(pair[0]==0)
            print("  Set B[{}, {}] to {}".format(i, i-1, B[i, i-1]))
        if i != N_b - 2: # c_i
            print("  Setting c_i for i =", i)
            B[i, i+1] = -1 * int(pair[1]==0)
            print("  Set B[{}, {}] to {}".format(i, i+1, B[i, i+1]))
        # b_i
        print("  Setting b_i for i =", i)
        B[i, i] = (int(pair[0]==0) or int(pair[1]==0)) + (int(pair[0]==0) and int(pair[1]==0))
        print("  Set B[{}, {}] to {}".format(i, i, B[i, i]))

    return B

Much better, this takes care of each diagonal using some boolean logic, each pair encodes the element at index $i+1$ and $i+2$.

In [111]:
generate_B(4, 1, 4)

Extracted n_s: [0 0 1 0]
Setting row 0 for pair (0, 0)
  Setting c_i for i = 0
  Set B[0, 1] to -1.0
  Setting b_i for i = 0
  Set B[0, 0] to 2.0
Setting row 1 for pair (0, 1)
  Setting a_i for i = 1
  Set B[1, 0] to -1.0
  Setting c_i for i = 1
  Set B[1, 2] to 0.0
  Setting b_i for i = 1
  Set B[1, 1] to 1.0
Setting row 2 for pair (1, 0)
  Setting a_i for i = 2
  Set B[2, 1] to 0.0
  Setting b_i for i = 2
  Set B[2, 2] to 1.0


array([[ 2., -1.,  0.],
       [-1.,  1.,  0.],
       [ 0.,  0.,  1.]])